# Manual Reconciliation Loop

**Prerequisites:**
- `docker compose up -d db minio minio-init` (db, minio running)
- `twod-fim-jobs:build_model` container built

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path
from urllib.parse import quote_plus
import geopandas as gpd

# Add scripts/ to path
sys.path.insert(0, str(Path("..").resolve()))

from recon.state_store import StateStore
from recon.config import settings
from recon.workers import LocalDockerRunner
from recon.reconciliation import run_and_update
from scripts.seed import load_network, seed


In [ ]:
store = StateStore()

# Container db_uri uses the Docker network hostname, not localhost
container_db_uri = (
    f"postgresql://{quote_plus(settings.postgres_user)}:{quote_plus(settings.postgres_password)}"
    f"@db:{settings.postgres_port}/{settings.postgres_db}"
)

print(f"DB (host):      {settings.postgres_host}:{settings.postgres_port}/{settings.postgres_db}")
print(f"DB (container): db:{settings.postgres_port}/{settings.postgres_db}")
print(f"Artifacts:      s3://{settings.artifacts_s3_bucket}")


In [ ]:
# Build the runner (same config as the Dagster resource)
# Container env vars use Docker service names, not localhost
env_vars = {
    "AWS_ENDPOINT_URL": "http://minio:9000",
}
for key in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN", "AWS_REQUEST_PAYER"):
    val = os.environ.get(key)
    if val:
        env_vars[key] = val

volumes = []
if settings.docker_data_dir:
    volumes.append(f"{settings.docker_data_dir}:/data:ro")

runner = LocalDockerRunner(
    image=settings.build_model_image,
    network=settings.docker_network,
    env_vars=env_vars,
    timeout=settings.build_model_timeout,
    platform=settings.docker_platform,
    volumes=volumes,
)
print(f"Runner: {runner.image} on network {runner.network}")
print(f"Container S3:  {env_vars['AWS_ENDPOINT_URL']}")


## 2. Seed reaches from GeoPackage

In [ ]:
# Reset all data tables (optional - use for a clean run)
store.reset()
print("All data tables cleared.")


In [ ]:
gpkg_path = Path("../testdata/network.gpkg")
network = load_network(gpkg_path)
print(f"Loaded {len(network)} reaches from {gpkg_path}")
for r in network:
    flags = []
    if r["is_terminal"]: flags.append("terminal")
    if r["is_headwater"]: flags.append("headwater")
    print(f"  {r['reach_id']} -> {r['reach_to_id']} {' '.join(flags)}")


In [ ]:
seed(store, network)
print(f"Seeded {len(network)} reaches.")


## 3. Query eligible reaches

This is the same query the Dagster reconciliation sensor runs.
The DB decides which reaches need work (downstream-first).

In [ ]:
eligible = store.get_eligible_reaches()
print(f"{len(eligible)} reaches eligible for build_model:")
for r in eligible:
    print(f"  reach {r['reach_id']} (revision {r['revision']}, terminal={r['is_terminal']})")


## 4. Run build_model for a single reach

Pick a reach from the eligible list and run the container.

In [ ]:
target = eligible[0]
reach_id = target["reach_id"]
revision = target["revision"]
print(f"Running build_model for reach {reach_id} (revision {revision})")


In [ ]:
result = run_and_update(
    reach_id=reach_id,
    revision=revision,
    runner=runner,
    store=store,
    db_uri=container_db_uri,
    lulc_source=settings.lulc_source,
)
print(f"model_id:      {result['model_id']}")
print(f"identity_hash: {result['identity_hash']}")
print(f"version:       {result['build_model_version']}")


## 5. Check reconciliation status

In [ ]:
states = store.get_reach_states()
for s in states:
    status = "reconciled" if s["reconciled"] else "pending"
    model = s["model_id"] or "-"
    print(f"  reach {s['reach_id']}: {status}  model_id={model}")

reconciled = sum(1 for s in states if s["reconciled"])
print(f"\n{reconciled}/{len(states)} reconciled")


## 6. Process all eligible reaches (background)

Runs in a background thread so you can re-run the map cell (section 7) to watch progress.
Output prints inline as reaches complete.

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

MAX_WORKERS = 10
MAX_RETRIES = 1

failures: dict[int, int] = {}
skipped: set[int] = set()
processing_done = threading.Event()

def process_reach(target):
    return run_and_update(
        reach_id=target["reach_id"],
        revision=target["revision"],
        runner=runner,
        store=store,
        db_uri=container_db_uri,
        lulc_source=settings.lulc_source,
    )

def run_all():
    wave = 0
    while True:
        eligible = store.get_eligible_reaches()
        eligible = [t for t in eligible if t["reach_id"] not in skipped]
        if not eligible:
            print("No more eligible reaches.")
            break

        wave += 1
        print(f"\n--- Wave {wave}: {len(eligible)} reaches (max {MAX_WORKERS} concurrent) ---")

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
            futures = {pool.submit(process_reach, t): t for t in eligible}
            for future in as_completed(futures):
                target = futures[future]
                reach_id = target["reach_id"]
                try:
                    result = future.result()
                    print(f"  reach {reach_id}: OK ({result['model_id']})")
                    failures.pop(reach_id, None)
                except RuntimeError as e:
                    failures[reach_id] = failures.get(reach_id, 0) + 1
                    if failures[reach_id] >= MAX_RETRIES:
                        print(f"  reach {reach_id}: FAILED ({MAX_RETRIES}/{MAX_RETRIES} retries exhausted, skipping)")
                        skipped.add(reach_id)
                    else:
                        print(f"  reach {reach_id}: FAILED ({failures[reach_id]}/{MAX_RETRIES}) - {str(e)[:150]}")

    states = store.get_reach_states()
    reconciled = sum(1 for s in states if s["reconciled"])
    print(f"\nDone: {reconciled}/{len(states)} reconciled")
    if skipped:
        print(f"{len(skipped)} skipped after {MAX_RETRIES} retries: {skipped}")
    processing_done.set()

thread = threading.Thread(target=run_all, daemon=True)
thread.start()
print("Processing started in background. Re-run the map cell (section 7) to watch progress.")

## 7. Network Map

Re-run this cell anytime to see current processing status on the map.

In [ ]:
import matplotlib.pyplot as plt
from shapely.wkt import loads as load_wkt

# Load geometry from the network (already in memory from seeding)
geom_by_id = {r["reach_id"]: load_wkt(r["geom"]) for r in network}

# Get current status from DB
states = store.get_reach_states()
state_by_id = {s["reach_id"]: s["reconciled"] for s in states}

# Get eligible reaches
eligible_ids = {r["reach_id"] for r in store.get_eligible_reaches()}

# Failed reaches (if concurrent loop has run)
failed_ids = skipped if "skipped" in dir() else set()

# Build GeoDataFrame for plotting
rows = []
for rid, geom in geom_by_id.items():
    if state_by_id.get(rid):
        status = "reconciled"
    elif rid in failed_ids:
        status = "failed"
    elif rid in eligible_ids:
        status = "eligible"
    else:
        status = "pending"
    rows.append({"reach_id": rid, "status": status, "geometry": geom})

map_gdf = gpd.GeoDataFrame(rows, crs="EPSG:5070")

colors = {"reconciled": "#2D6A4F", "eligible": "#E67E22", "failed": "#C0392B", "pending": "#95A5A6"}
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
for status, color in colors.items():
    subset = map_gdf[map_gdf["status"] == status]
    if not subset.empty:
        subset.plot(ax=ax, color=color, linewidth=2, label=f"{status} ({len(subset)})")

ax.legend(loc="upper left", fontsize=9)
ax.set_title(f"Network Status: {sum(1 for s in states if s['reconciled'])}/{len(states)} reconciled", fontsize=12)
ax.set_axis_off()
plt.tight_layout()
plt.show()
